# Extinction-value experiments

This notebook covers the complete `mmcfilters` extinction workflow: ranking extrema, extinction filtering, cutoff-contour visualization, the dense extinction valuation, and formal edge-indexed saliency maps.

Tree construction, extinction, filtering, LCA projection, hierarchical-watershed persistence, and threshold cuts are executed by the C++ library through its Python bindings. Python prepares inputs and formats tables and figures. The extinction traversal follows Alexandre Gonçalves Silva and Roberto de Alencar Lotufo, [*Efficient computation of new extinction values from extended component tree*](https://doi.org/10.1016/j.patrec.2010.07.019), *Pattern Recognition Letters* 32(1), 79–90, 2011. The formal watershed follows Jean Cousty, Laurent Najman, Yukiko Kenmochi, and Silvio Guimarães, [*Hierarchical segmentations with graphs: quasi-flat zones, minimum spanning trees, and saliency maps*](https://doi.org/10.1007/s10851-017-0768-7), *Journal of Mathematical Imaging and Vision* 60(4), 479–502, 2018. Exact mappings and library adaptations are in [`docs/saliency.md`](../docs/saliency.md#primary-references-and-implementation-correspondence).

## Goals

- Build a max-tree and min-tree from the same image.
- Compare `AREA` and `LEVEL` as extinction-strength attributes.
- Inspect the C++ `(leaf, cutoffNode, extinction)` records.
- Filter through explicit top-k and minimum-extinction policies.
- Display `contourMap(...)` as a raster cutoff-contour visualization.
- Compare the dense extinction valuation, Cousty persistence map, and former monotone LCA projection.
- Materialize edge cuts with `HierarchySaliencyMapProjection.thresholdCut(...)`.

## Setup

The notebook imports the installed `mmcfilters` package directly. Prepare the environment outside the notebook as described in `README.md`.

In [ ]:
from __future__ import annotations

import mmcfilters
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage import data

%matplotlib inline
print(f"mmcfilters: {mmcfilters.__version__}")


## Parameters and image

`CROP` can reduce the cost of edge maps and make figures easier to read. The default uses the complete `skimage.data.camera()` image.

In [ ]:
CROP = None  # Example: (slice(64, 192), slice(64, 192))
RADIUS = 1.5
KEEP_COUNTS = [1, 2, 4, 8, 16, 32]
ATTRIBUTE_NAME = "AREA"

image = np.ascontiguousarray(data.camera(), dtype=np.uint8)
if CROP is not None:
    image = np.ascontiguousarray(image[CROP])
print("shape:", image.shape, "dtype:", image.dtype, "min/max:", int(image.min()), int(image.max()))

plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="gray")
plt.title("Input image: skimage.data.camera()")
plt.axis("off");


## Build trees and attributes

`ExtinctionValues` accepts max-trees and min-trees because both declare a globally monotone altitude order. We compute `AREA` and `LEVEL` to compare two definitions of extremum strength.

In [ ]:
max_tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(image, radius=RADIUS)
min_tree = mmcfilters.MorphologicalTreeFactory.createMinTree(image, radius=RADIUS)

attributes = {
    "AREA": mmcfilters.Attribute.AREA,
    "LEVEL": mmcfilters.Attribute.LEVEL,
}

experiments = {}
for tree_name, tree in [("max-tree", max_tree), ("min-tree", min_tree)]:
    experiments[tree_name] = {"tree": tree}
    for attribute_name, attribute_id in attributes.items():
        values = mmcfilters.Attribute.computeSingleAttribute(tree, attribute_id)
        extinction = mmcfilters.ExtinctionValues(tree, values)
        experiments[tree_name][attribute_name] = {
            "attribute": values,
            "extinction": extinction,
            "records": extinction.getRegionalExtrema(),
        }

summary_rows = []
for tree_name, entry in experiments.items():
    tree = entry["tree"]
    for attribute_name in attributes:
        records = entry[attribute_name]["records"]
        summary_rows.append({
            "tree": tree_name,
            "attribute": attribute_name,
            "nodes": int(tree.numInternalNodeSlots),
            "alive_nodes": len(tree.aliveNodeIds),
            "extrema": len(records),
            "dominant_value": records[0][2] if records else np.nan,
        })

pd.DataFrame(summary_rows)

## Extinction records

Each record comes directly from `ExtinctionValues.getRegionalExtrema()` as `(leafNodeId, cutoffNodeId, value)`, sorted by decreasing extinction, then deterministic node-id tie breaks.

In [ ]:
def extinction_records_frame(tree_name: str, attribute_name: str, limit: int = 12) -> pd.DataFrame:
    records = experiments[tree_name][attribute_name]["records"]
    rows = []
    for rank, (leaf, cutoff, value) in enumerate(records[:limit], start=1):
        rows.append({
            "rank": rank,
            "leaf": int(leaf),
            "cutoffNode": int(cutoff),
            "extinction": float(value),
        })
    return pd.DataFrame(rows)


for tree_name in ["max-tree", "min-tree"]:
    print(tree_name, ATTRIBUTE_NAME)
    display(extinction_records_frame(tree_name, ATTRIBUTE_NAME, limit=10))

## Extinction filtering

`filtering(selection)` reconstructs the image under an explicit policy. `ExtinctionSelectionPolicy.byTopK(k)` retains the first `k` ranked extrema; `byThreshold(t)` retains every extremum with `extinction >= t`.

In [ ]:
def show_image_grid(title: str, images: list[tuple[str, np.ndarray]], cmap: str = "gray", columns: int = 4) -> None:
    rows = int(np.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(3.2 * columns, 3.2 * rows), squeeze=False)
    fig.suptitle(title)
    for ax, (label, img) in zip(axes.ravel(), images):
        ax.imshow(img, cmap=cmap)
        ax.set_title(label)
        ax.axis("off")
    for ax in axes.ravel()[len(images):]:
        ax.axis("off")
    plt.tight_layout()


filtering_images = [("original", image)]
for keep in KEEP_COUNTS:
    filtered = experiments["max-tree"][ATTRIBUTE_NAME]["extinction"].filtering(mmcfilters.ExtinctionSelectionPolicy.byTopK(keep))
    filtering_images.append((f"keep={keep}", filtered))
show_image_grid(f"Extinction filtering: max-tree / {ATTRIBUTE_NAME}", filtering_images, columns=4)

records = experiments["max-tree"][ATTRIBUTE_NAME]["records"]
candidate_values = np.array([float(value) for _, _, value in records[1:] if np.isfinite(value)], dtype=float)
if candidate_values.size:
    thresholds = np.unique(np.quantile(candidate_values, [0.99, 0.999, 0.99999]))
else:
    thresholds = np.array([float(records[0][2])], dtype=float)

threshold_images = [("original", image)]
for threshold in thresholds:
    filtered = experiments["max-tree"][ATTRIBUTE_NAME]["extinction"].filtering(mmcfilters.ExtinctionSelectionPolicy.byThreshold(float(threshold)))
    threshold_images.append((f"thr≥{threshold:.3g}", filtered))
show_image_grid(f"Filtering by threshold: max-tree / {ATTRIBUTE_NAME}", threshold_images, columns=4)

## Cutoff-contour visualization

`contourMap(selection, scorePolicy)` draws selected cutoff contours in the image domain. It is a visualization, not the formal edge-indexed saliency map of a hierarchy.

In [ ]:
legacy_maps = []
for keep in [4, 8, 16, 32]:
    saliency = experiments["max-tree"][ATTRIBUTE_NAME]["extinction"].contourMap(mmcfilters.ExtinctionSelectionPolicy.byTopK(keep), mmcfilters.ExtinctionContourScorePolicy.RankScore)
    legacy_maps.append((f"keep={keep}", saliency))
show_image_grid(f"contourMap: max-tree / {ATTRIBUTE_NAME}", legacy_maps, cmap="gray", columns=4)

## Dense extinction valuation

`getExtinctionValueAttribute()` returns the cached max-descendant extinction valuation. Leaves receive their extinction and ancestors receive the maximum value in their subtree. It is a valid monotone valuation, but projecting it directly by LCA is the former behavior, now exposed as `computeMonotoneExtinctionProjection(...)`.

In [ ]:
def attribute_summary(tree_name: str, attribute_name: str) -> pd.DataFrame:
    tree = experiments[tree_name]["tree"]
    extinction = experiments[tree_name][attribute_name]["extinction"]
    raw_attribute = extinction.getExtinctionValueAttribute()
    ranked_attribute = extinction.computeRankedExtinctionValueAttribute()
    mmcfilters.HierarchySaliencyMapValidation.validateHierarchyValuation(tree, raw_attribute, nonnegative=True)
    mmcfilters.HierarchySaliencyMapValidation.validateHierarchyValuation(tree, ranked_attribute, nonnegative=True)
    return pd.DataFrame([
        {
            "tree": tree_name,
            "attribute": attribute_name,
            "dtype": str(raw_attribute.dtype),
            "raw_min": float(np.min(raw_attribute)),
            "raw_max": float(np.max(raw_attribute)),
            "ranked_levels": int(np.unique(ranked_attribute).size),
            "ranked_min": int(np.min(ranked_attribute)),
            "ranked_max": int(np.max(ranked_attribute)),
        }
    ])

pd.concat(
    [attribute_summary(tree_name, attribute_name) for tree_name in ["max-tree", "min-tree"] for attribute_name in attributes],
    ignore_index=True,
)

## Formal edge saliency

`computeFormalSaliencyEdgeMap(ranked=True)` executes the Cousty persistence MST/BPTAO construction and returns `sources`, `targets`, and `values`. It is not a direct LCA projection of the dense extinction valuation. The figures draw high-valued returned edges over the image.

In [ ]:
def edge_map_summary(name: str, edge_map: dict) -> dict:
    values = np.asarray(edge_map["values"])
    return {
        "name": name,
        "edges": int(values.size),
        "dtype": str(values.dtype),
        "min": float(values.min()) if values.size else np.nan,
        "p50": float(np.percentile(values, 50)) if values.size else np.nan,
        "p90": float(np.percentile(values, 90)) if values.size else np.nan,
        "max": float(values.max()) if values.size else np.nan,
    }

edge_maps = {}
rows = []
for tree_name in ["max-tree", "min-tree"]:
    extinction = experiments[tree_name][ATTRIBUTE_NAME]["extinction"]
    edge_map = extinction.computeFormalSaliencyEdgeMap(ranked=True)
    edge_maps[tree_name] = edge_map
    rows.append(edge_map_summary(tree_name, edge_map))

pd.DataFrame(rows)

In [ ]:
def edge_segments(edge_map: dict) -> np.ndarray:
    rows = int(edge_map["numRows"])
    cols = int(edge_map["numCols"])
    source = np.asarray(edge_map["sources"], dtype=np.int64)
    target = np.asarray(edge_map["targets"], dtype=np.int64)
    source_xy = np.column_stack((source % cols, source // cols))
    target_xy = np.column_stack((target % cols, target // cols))
    return np.stack((source_xy, target_xy), axis=1)


def show_edge_saliency(title: str, base_image: np.ndarray, edge_map: dict, quantile: float = 0.90) -> None:
    values = np.asarray(edge_map["values"])
    threshold = np.quantile(values, quantile)
    mask = values >= threshold
    segments = edge_segments(edge_map)[mask]
    weights = values[mask]

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.imshow(base_image, cmap="gray")
    collection = LineCollection(segments, array=weights, cmap="magma", linewidths=1.0, alpha=0.95)
    ax.add_collection(collection)
    fig.colorbar(collection, ax=ax, fraction=0.046, pad=0.04, label="ranked saliency")
    ax.set_title(f"{title} | edges >= p{int(100 * quantile)}")
    ax.set_xlim(-0.5, base_image.shape[1] - 0.5)
    ax.set_ylim(base_image.shape[0] - 0.5, -0.5)
    ax.axis("off")
    plt.tight_layout()


for tree_name, edge_map in edge_maps.items():
    show_edge_saliency(f"Formal edge map: {tree_name} / {ATTRIBUTE_NAME}", image, edge_map, quantile=0.92)

## Threshold cuts

`HierarchySaliencyMapProjection.thresholdCut(edge_map, threshold)` returns edges with saliency at least the threshold. Ranked maps use integer thresholds.

In [ ]:
def show_threshold_cut(title: str, base_image: np.ndarray, edge_map: dict, threshold: float) -> dict:
    cut = mmcfilters.HierarchySaliencyMapProjection.thresholdCut(edge_map, threshold=threshold)
    segments = edge_segments(cut)

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.imshow(base_image, cmap="gray")
    if len(segments):
        ax.add_collection(LineCollection(segments, colors="#f03b20", linewidths=1.0, alpha=0.95))
    ax.set_title(f"{title} | threshold={threshold:g} | edges={len(segments)}")
    ax.set_xlim(-0.5, base_image.shape[1] - 0.5)
    ax.set_ylim(base_image.shape[0] - 0.5, -0.5)
    ax.axis("off")
    plt.tight_layout()
    return cut


max_values = np.asarray(edge_maps["max-tree"]["values"])
thresholds = [int(np.quantile(max_values, q)) for q in [0.75, 0.90, 0.97]]
thresholds = sorted(set(thresholds))
cut_rows = []
for threshold in thresholds:
    cut = show_threshold_cut("Corte formal: max-tree / AREA", image, edge_maps["max-tree"], threshold)
    cut_rows.append({"threshold": threshold, "cut_edges": int(len(cut["sources"]))})

pd.DataFrame(cut_rows)

## Compare input attributes

The same tree can use different node attributes to measure extremum strength. Here `AREA` and `LEVEL` are compared on the ranked persistence saliency map.

In [ ]:
comparison_rows = []
for attribute_name in ["AREA", "LEVEL"]:
    extinction = experiments["max-tree"][attribute_name]["extinction"]
    ranked_attribute = extinction.computeRankedExtinctionValueAttribute()
    edge_map = extinction.computeFormalSaliencyEdgeMap(ranked=True)
    comparison_rows.append({
        "attribute": attribute_name,
        "ranked_node_levels": int(np.unique(ranked_attribute).size),
        "edge_levels": int(np.unique(edge_map["values"]).size),
        "edge_max": int(np.max(edge_map["values"])),
        "edge_p90": float(np.percentile(edge_map["values"], 90)),
    })

pd.DataFrame(comparison_rows)

## Final checks

- `filtering` and `contourMap` are image-domain APIs controlled by a selection policy.
- `computeFormalSaliencyEdgeMap` is the Cousty persistence path on graph edges.
- `computeMonotoneExtinctionProjection` names the former max-descendant-plus-LCA behavior.
- `thresholdCut` operates on the edge-indexed formal map.

Change `ATTRIBUTE_NAME`, `KEEP_COUNTS`, `CROP`, and `RADIUS` for further experiments.